# Imports

In [1]:
import glob
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.model_selection import cross_val_score
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from colorama import Fore, Style
import warnings
warnings.filterwarnings("ignore")

# Data Loading

In [2]:
def PrintColor(text, color=Fore.CYAN):
    print(color + Style.BRIGHT + text + Style.RESET_ALL)

DATA_PATH = '/kaggle/input/competitions/march-machine-learning-mania-2025/'
# ============================================================
# 1. LOAD RAW DATA
# ============================================================
PrintColor("---> 1. Loading Data...")

reg_comp     = pd.concat([pd.read_csv(DATA_PATH + "MRegularSeasonCompactResults.csv"),
                          pd.read_csv(DATA_PATH + "WRegularSeasonCompactResults.csv")], ignore_index=True)
reg_det      = pd.concat([pd.read_csv(DATA_PATH + "MRegularSeasonDetailedResults.csv"),
                          pd.read_csv(DATA_PATH + "WRegularSeasonDetailedResults.csv")], ignore_index=True)
tourney_comp = pd.concat([pd.read_csv(DATA_PATH + "MNCAATourneyCompactResults.csv"),
                          pd.read_csv(DATA_PATH + "WNCAATourneyCompactResults.csv")], ignore_index=True)
tourney_det  = pd.concat([pd.read_csv(DATA_PATH + "MNCAATourneyDetailedResults.csv"),
                          pd.read_csv(DATA_PATH + "WNCAATourneyDetailedResults.csv")], ignore_index=True)
seeds_raw    = pd.concat([pd.read_csv(DATA_PATH + "MNCAATourneySeeds.csv"),
                          pd.read_csv(DATA_PATH + "WNCAATourneySeeds.csv")], ignore_index=True)
conferences  = pd.concat([pd.read_csv(DATA_PATH + "MTeamConferences.csv"),
                          pd.read_csv(DATA_PATH + "WTeamConferences.csv")], ignore_index=True)

# Seeds dict for fast lookup
seeds_dict = {
    '_'.join(map(str, [int(k1), k2])): int(v[1:3])
    for k1, v, k2 in seeds_raw[['Season', 'Seed', 'TeamID']].values
}

# Seeds df for merging
seeds_df = seeds_raw.copy()
seeds_df['seed_num'] = seeds_df['Seed'].str.extract(r'(\d+)').astype(float)
seeds_df = seeds_df[['Season', 'TeamID', 'seed_num']].dropna()

---> 1. Loading Data...


# Feature Extraction

In [3]:
PrintColor("---> 2. Engineering Team Features...")

# A. Dynamic Elo
def compute_elo_ratings(df):
    df = df.sort_values(["Season", "DayNum"]).reset_index(drop=True)
    elo, season_elo = {}, {}
    prev_season = None
    for _, row in df.iterrows():
        s, w, l = row["Season"], row["WTeamID"], row["LTeamID"]
        loc, w_pts, l_pts = row.get("WLoc", "N"), row["WScore"], row["LScore"]
        if s != prev_season:
            if prev_season is not None:
                for t, r in elo.items(): season_elo[(prev_season, t)] = r
                for t in elo: elo[t] = 1500 + 0.75 * (elo[t] - 1500)
            prev_season = s
        if w not in elo: elo[w] = 1500
        if l not in elo: elo[l] = 1500
        if row["DayNum"] <= 132:
            w_adj = elo[w] + 100 if loc == "H" else elo[w]
            l_adj = elo[l] + 100 if loc == "A" else elo[l]
            w_exp = 1.0 / (1.0 + 10.0 ** ((l_adj - w_adj) / 400.0))
            mov  = np.log(max(abs(w_pts - l_pts), 1) + 1) * (2.2 / ((elo[w] - elo[l]) * 0.001 + 2.2))
            elo[w] += (32 * mov) * (1 - w_exp)
            elo[l] -= (32 * mov) * (1 - w_exp)
    if prev_season:
        for t, r in elo.items(): season_elo[(prev_season, t)] = r
    return pd.DataFrame([(s, t, e) for (s, t), e in season_elo.items()],
                        columns=["Season", "TeamID", "elo"])

elo_df = compute_elo_ratings(reg_comp)

# B. Conference Strength
conf_merged = pd.merge(conferences, elo_df, on=["Season", "TeamID"], how="left").fillna(1500)
conf_str = conf_merged.groupby(["Season", "ConfAbbrev"])['elo'].mean().reset_index(name='conf_elo')
conf_df = pd.merge(conferences, conf_str, on=["Season", "ConfAbbrev"], how="left")[["Season", "TeamID", "conf_elo"]]

# C. Advanced Efficiency Stats
def compute_all_stats(df):
    w_cols = ['WScore','WFGM','WFGA','WFGM3','WFGA3','WFTM','WFTA','WOR','WDR','WAst','WTO','WStl','WBlk','WPF']
    l_cols = ['LScore','LFGM','LFGA','LFGM3','LFGA3','LFTM','LFTA','LOR','LDR','LAst','LTO','LStl','LBlk','LPF']
    w_df = df[['Season','WTeamID'] + w_cols].rename(columns=lambda x: x.replace('W','') if x.startswith('W') else x)
    l_df = df[['Season','LTeamID'] + l_cols].rename(columns=lambda x: x.replace('L','') if x.startswith('L') else x)
    stat_df = pd.concat([w_df, l_df], ignore_index=True).rename(columns={'eamID':'TeamID'})
    stat_df['Poss']      = stat_df['FGA'] + 0.475*stat_df['FTA'] - stat_df['OR'] + stat_df['TO']
    stat_df['eFG']       = (stat_df['FGM'] + 0.5*stat_df['FGM3']) / stat_df['FGA'].replace(0,1)
    stat_df['TOR']       = stat_df['TO'] / stat_df['Poss'].replace(0,1)
    stat_df['OffRating'] = 100 * (stat_df['Score'] / stat_df['Poss'].replace(0,1))
    stat_df['FT_rate']   = stat_df['FTA'] / stat_df['FGA'].replace(0,1)
    stat_df['3PA_rate']  = stat_df['FGA3'] / stat_df['FGA'].replace(0,1)
    agg_cols = [c for c in stat_df.columns if c not in ['Season','TeamID']]
    return stat_df.groupby(['Season','TeamID'])[agg_cols].mean().reset_index()

all_stats_df = compute_all_stats(reg_det)

# D. Massey/KenPom Ordinals
def extract_massey(path, system='KPK'):
    try:
        ordinals = pd.read_csv(path + "MMasseyOrdinals.csv")
        if 'RankingDayNum' in ordinals.columns:
            ordinals.rename(columns={'RankingDayNum':'DayNum'}, inplace=True)
        sys_data = ordinals[ordinals['SystemName'] == system]
        idx = sys_data.groupby(['Season','TeamID'])['DayNum'].idxmax()
        return sys_data.loc[idx, ['Season','TeamID','OrdinalRank']].rename(columns={'OrdinalRank': f'{system}_rank'})
    except:
        return pd.DataFrame(columns=['Season','TeamID', f'{system}_rank'])

kp_df = extract_massey(DATA_PATH, 'KPK')

# E. Late-Season Momentum (last 14 days ~DayNum >= 118)
late_reg = reg_comp[reg_comp['DayNum'] >= 118].copy()
w_mom = late_reg.groupby(['Season','WTeamID']).size().reset_index(name='W').rename(columns={'WTeamID':'TeamID'})
l_mom = late_reg.groupby(['Season','LTeamID']).size().reset_index(name='L').rename(columns={'LTeamID':'TeamID'})
mom_df = pd.merge(w_mom, l_mom, on=['Season','TeamID'], how='outer').fillna(0)
mom_df['win_ratio_14d'] = mom_df['W'] / (mom_df['W'] + mom_df['L'])
mom_df = mom_df[['Season','TeamID','win_ratio_14d']]

# F. Win % + Avg Margin (full season)
w_all = reg_comp.groupby(['Season','WTeamID']).agg(wins=('WScore','count'), w_margin=('WScore', lambda x: (x - reg_comp.loc[x.index,'LScore']).mean())).reset_index().rename(columns={'WTeamID':'TeamID'})
l_all = reg_comp.groupby(['Season','LTeamID']).agg(losses=('LScore','count')).reset_index().rename(columns={'LTeamID':'TeamID'})
wl_df = pd.merge(w_all, l_all, on=['Season','TeamID'], how='outer').fillna(0)
wl_df['win_pct'] = wl_df['wins'] / (wl_df['wins'] + wl_df['losses'])
wl_df = wl_df[['Season','TeamID','win_pct','w_margin']]

# --- Unified Team Features ---
team_features = elo_df\
    .merge(all_stats_df,  on=['Season','TeamID'], how='left')\
    .merge(conf_df,       on=['Season','TeamID'], how='left')\
    .merge(kp_df,         on=['Season','TeamID'], how='left')\
    .merge(mom_df,        on=['Season','TeamID'], how='left')\
    .merge(wl_df,         on=['Season','TeamID'], how='left')\
    .merge(seeds_df,      on=['Season','TeamID'], how='left')

team_features['KPK_rank'] = team_features['KPK_rank'].fillna(100)
team_features['seed_num'] = team_features['seed_num'].fillna(20)
team_features.fillna(0, inplace=True)


---> 2. Engineering Team Features...


In [4]:
PrintColor("---> 3. Building Head-to-Head Aggregated Features...")

all_games = pd.concat([reg_det, tourney_det], ignore_index=True)
all_games['IDTeams'] = all_games.apply(
    lambda r: '_'.join(map(str, sorted([r['WTeamID'], r['LTeamID']]))), axis=1)

c_score_col = [
    'NumOT','WFGM','WFGA','WFGM3','WFGA3','WFTM','WFTA','WOR','WDR','WAst',
    'WTO','WStl','WBlk','WPF','LFGM','LFGA','LFGM3','LFGA3','LFTM','LFTA',
    'LOR','LDR','LAst','LTO','LStl','LBlk','LPF'
]
c_score_agg = ['sum','mean','median','max','min','std','skew','nunique']
gb = all_games.groupby("IDTeams").agg({k: c_score_agg for k in c_score_col}).reset_index()
gb.columns = ["".join(c) + "_h2h" if c[0] != "IDTeams" else "IDTeams" for c in gb.columns]
gb_col_name = "IDTeams"  # already named properly after renaming

# Fix column naming (MultiIndex flatten)
gb.columns = [f"{a}_{b}_h2h" if b else a for a, b in gb.columns] if isinstance(gb.columns, pd.MultiIndex) else gb.columns


---> 3. Building Head-to-Head Aggregated Features...


In [5]:
PrintColor("---> 4. Building Matchup Difference Features...")

def build_matchups(df, team_features, gb):
    df = df.copy()
    df['T1_TeamID'] = np.minimum(df['WTeamID'], df['LTeamID'])
    df['T2_TeamID'] = np.maximum(df['WTeamID'], df['LTeamID'])
    df['target']    = (df['WTeamID'] == df['T1_TeamID']).astype(int)

    # Attach team-level features
    tf_cols = [c for c in team_features.columns if c != 'TeamID']
    df = df.merge(team_features.rename(columns={c: f'T1_{c}' for c in team_features.columns if c not in ['Season','TeamID']}),
                  left_on=['Season','T1_TeamID'], right_on=['Season','TeamID'], how='left').drop(columns=['TeamID'], errors='ignore')
    df = df.merge(team_features.rename(columns={c: f'T2_{c}' for c in team_features.columns if c not in ['Season','TeamID']}),
                  left_on=['Season','T2_TeamID'], right_on=['Season','TeamID'], how='left').drop(columns=['TeamID'], errors='ignore')

    feat_base = [c for c in team_features.columns if c not in ['Season','TeamID']]
    diff_cols = []
    for col in feat_base:
        df[f'{col}_diff'] = df[f'T1_{col}'] - df[f'T2_{col}']
        diff_cols.append(f'{col}_diff')

    # Seed ratio feature
    df['seed_ratio'] = df['T1_seed_num'] / (df['T2_seed_num'] + 1)
    diff_cols.append('seed_ratio')

    # H2H aggregated features
    df['IDTeams'] = df.apply(lambda r: '_'.join(map(str, sorted([r['T1_TeamID'], r['T2_TeamID']]))), axis=1)
    h2h_cols = [c for c in gb.columns if c != 'IDTeams']
    df = df.merge(gb, on='IDTeams', how='left')
    all_feature_cols = diff_cols + h2h_cols

    return df[['Season','T1_TeamID','T2_TeamID','target'] + all_feature_cols].fillna(0), diff_cols, h2h_cols

train_raw = pd.concat([
    tourney_comp,
    reg_comp[reg_comp['DayNum'] >= 100].sample(frac=0.15, random_state=42)
], ignore_index=True)

train_data, diff_features, h2h_features = build_matchups(train_raw, team_features, gb)
train_data = train_data[train_data['Season'] >= 2010].reset_index(drop=True)
all_features = diff_features + h2h_features


---> 4. Building Matchup Difference Features...


# Filteration
- using vif 

In [6]:
PrintColor("---> 5. VIF & OLS Filtering on Diff Features...")

X_sample = sm.add_constant(train_data[diff_features].sample(min(3000, len(train_data)), random_state=42))
final_diff_features = diff_features.copy()

while True:
    vifs = [variance_inflation_factor(X_sample.values, i) for i in range(X_sample.shape[1])]
    max_vif = max(vifs[1:])
    if max_vif > 15.0:
        idx = vifs.index(max_vif)
        drop_col = X_sample.columns[idx]
        X_sample = X_sample.drop(columns=[drop_col])
        if drop_col in final_diff_features:
            final_diff_features.remove(drop_col)
    else:
        break

ols_mod = sm.OLS(train_data['target'], sm.add_constant(train_data[final_diff_features])).fit()
sig_features = ols_mod.pvalues[ols_mod.pvalues < 0.05].index.tolist()
if 'const' in sig_features: sig_features.remove('const')
final_diff_features = sig_features if len(sig_features) >= 5 else final_diff_features

# Final feature set = filtered diffs + all h2h
final_features = final_diff_features + h2h_features
PrintColor(f"   Total features after filtering: {len(final_features)}", color=Fore.GREEN)


---> 5. VIF & OLS Filtering on Diff Features...
   Total features after filtering: 222


# Model definition

In [7]:
PrintColor("---> 6. Training Ensemble (XGB + LGBM + CAT + ExtraTrees)...")

imputer = SimpleImputer(strategy='mean')
scaler  = StandardScaler()

X_all_imp = imputer.fit_transform(train_data[final_features])
X_all_sc  = scaler.fit_transform(X_all_imp)

val_seasons = [2024, 2023, 2022]
models_dict = {'xgb': [], 'lgb': [], 'cat': [], 'et': []}

for val_year in val_seasons:
    print(f"\n--- Validating on {val_year} ---")
    tr = train_data[train_data['Season'] != val_year]
    va = train_data[train_data['Season'] == val_year]

    tr_idx = tr.index.tolist()
    va_idx = va.index.tolist()

    X_tr = X_all_sc[tr_idx]; y_tr = tr['target'].values
    X_va = X_all_sc[va_idx]; y_va = va['target'].values

    # XGBoost
    xgb_m = xgb.XGBClassifier(n_estimators=600, learning_rate=0.01, max_depth=4,
                                subsample=0.8, colsample_bytree=0.8,
                                eval_metric='logloss', verbosity=0, random_state=42)
    xgb_m.fit(X_tr, y_tr)
    xgb_p = xgb_m.predict_proba(X_va)[:, 1]
    models_dict['xgb'].append(xgb_m)

    # LightGBM
    lgb_m = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.01, max_depth=4,
                                 subsample=0.8, colsample_bytree=0.8, verbose=-1, random_state=42)
    lgb_m.fit(X_tr, y_tr)
    lgb_p = lgb_m.predict_proba(X_va)[:, 1]
    models_dict['lgb'].append(lgb_m)

    # CatBoost
    cat_m = CatBoostClassifier(iterations=600, learning_rate=0.01, depth=4, verbose=0, random_seed=42)
    cat_m.fit(X_tr, y_tr)
    cat_p = cat_m.predict_proba(X_va)[:, 1]
    models_dict['cat'].append(cat_m)

    # ExtraTrees (Doc 2's champion)
    et_m = ExtraTreesClassifier(n_estimators=250, max_depth=19, min_samples_split=3,
                                 max_features='sqrt', random_state=42)
    et_m.fit(X_tr, y_tr)
    et_p = et_m.predict_proba(X_va)[:, 1]
    models_dict['et'].append(et_m)

    blend = np.clip((xgb_p + lgb_p + cat_p + et_p) / 4, 0.05, 0.95)
    print(f"  XGB   Brier: {brier_score_loss(y_va, xgb_p):.5f}")
    print(f"  LGBM  Brier: {brier_score_loss(y_va, lgb_p):.5f}")
    print(f"  CAT   Brier: {brier_score_loss(y_va, cat_p):.5f}")
    print(f"  ET    Brier: {brier_score_loss(y_va, et_p):.5f}")
    print(Fore.GREEN + f"  BLENDED Brier: {brier_score_loss(y_va, blend):.5f}" + Style.RESET_ALL)


---> 6. Training Ensemble (XGB + LGBM + CAT + ExtraTrees)...

--- Validating on 2024 ---
  XGB   Brier: 0.15034
  LGBM  Brier: 0.15038
  CAT   Brier: 0.15046
  ET    Brier: 0.18570
  BLENDED Brier: 0.15262

--- Validating on 2023 ---
  XGB   Brier: 0.14985
  LGBM  Brier: 0.14995
  CAT   Brier: 0.14872
  ET    Brier: 0.18225
  BLENDED Brier: 0.15193

--- Validating on 2022 ---
  XGB   Brier: 0.16281
  LGBM  Brier: 0.16345
  CAT   Brier: 0.16296
  ET    Brier: 0.18692
  BLENDED Brier: 0.16314


# Isotonic Data Calibration

In [8]:
PrintColor("---> 7. Isotonic Calibration...")

raw_train_preds = []
for i in range(len(val_seasons)):
    raw_train_preds.append(models_dict['xgb'][i].predict_proba(X_all_sc)[:, 1])
    raw_train_preds.append(models_dict['lgb'][i].predict_proba(X_all_sc)[:, 1])
    raw_train_preds.append(models_dict['cat'][i].predict_proba(X_all_sc)[:, 1])
    raw_train_preds.append(models_dict['et'][i].predict_proba(X_all_sc)[:, 1])

ensemble_train = np.mean(raw_train_preds, axis=0)
y_train_all = train_data['target'].values

ir_cal = IsotonicRegression(out_of_bounds="clip")
ir_cal.fit(ensemble_train, y_train_all)
cal_train = ir_cal.transform(ensemble_train)
print(f"  Calibrated Train Log Loss : {log_loss(y_train_all, cal_train):.5f}")
print(f"  Calibrated Train Brier    : {brier_score_loss(y_train_all, cal_train):.5f}")


---> 7. Isotonic Calibration...
  Calibrated Train Log Loss : 0.30901
  Calibrated Train Brier    : 0.10174


# Submission

In [9]:
PrintColor("---> 8. Generating Submission...")

sub = pd.read_csv(DATA_PATH + "SampleSubmissionStage2.csv")
sub['Season']   = sub['ID'].apply(lambda x: int(x.split('_')[0]))
sub['WTeamID']  = sub['ID'].apply(lambda x: int(x.split('_')[1]))
sub['LTeamID']  = sub['ID'].apply(lambda x: int(x.split('_')[2]))

sub_data, _, _ = build_matchups(sub, team_features, gb)
X_sub_raw = sub_data[final_features].fillna(0)
X_sub_imp = imputer.transform(X_sub_raw)
X_sub_sc  = scaler.transform(X_sub_imp)

all_sub_preds = []
for i in range(len(val_seasons)):
    all_sub_preds.append(models_dict['xgb'][i].predict_proba(X_sub_sc)[:, 1])
    all_sub_preds.append(models_dict['lgb'][i].predict_proba(X_sub_sc)[:, 1])
    all_sub_preds.append(models_dict['cat'][i].predict_proba(X_sub_sc)[:, 1])
    all_sub_preds.append(models_dict['et'][i].predict_proba(X_sub_sc)[:, 1])

ensemble_sub = np.mean(all_sub_preds, axis=0)
calibrated_preds = np.clip(ir_cal.transform(ensemble_sub), 0.05, 0.95)

sub['Pred'] = calibrated_preds
sub[['ID', 'Pred']].to_csv("submission_merged_pipeline.csv", index=False)

PrintColor("\n Pipeline Complete! submission_merged_pipeline.csv saved.", color=Fore.GREEN)

---> 8. Generating Submission...

 Pipeline Complete! submission_merged_pipeline.csv saved.
